1. Pytorch Version

In [5]:
import torch
print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch version: 2.13.0
Device: cpu


2. Imports

In [6]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


3. Device

In [7]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)

Using device: mps


4. Project Root

In [8]:
PROJECT_ROOT = Path.cwd().parent
print("Project root:", PROJECT_ROOT)

Project root: /Users/macbookpro/PycharmProjects/Techcart_ai


5. Dataset Path & File Check

In [9]:
TRAIN_PATH = PROJECT_ROOT / "data" / "splits" / "train.csv"
VAL_PATH = PROJECT_ROOT / "data" / "splits" / "validation.csv"
TEST_PATH = PROJECT_ROOT / "data" / "splits" / "test.csv"

print("Train:", TRAIN_PATH)
print("Val:", VAL_PATH)
print("Test:", TEST_PATH)
print("Train exists:", TRAIN_PATH.exists())
print("Val exists:", VAL_PATH.exists())
print("Test exists:", TEST_PATH.exists())

Train: /Users/macbookpro/PycharmProjects/Techcart_ai/data/splits/train.csv
Val: /Users/macbookpro/PycharmProjects/Techcart_ai/data/splits/validation.csv
Test: /Users/macbookpro/PycharmProjects/Techcart_ai/data/splits/test.csv
Train exists: True
Val exists: True
Test exists: True


6. Check Files & Columns

In [10]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train Shape:", train_df.shape)
print("Val Shape:", val_df.shape)
print("Test Shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())

Train Shape: (150, 7)
Val Shape: (19, 7)
Test Shape: (19, 7)
Columns: ['product_id', 'name', 'category', 'brand', 'price', 'stock', 'rating']


7. Preview Data

In [11]:
print(train_df.head())

   product_id                  name category     brand    price  stock  rating
0          38  G Pro X Superlight 2    Mouse  Logitech   153.56   21.0     4.8
1         103            1TB SN850X      SSD        WD    95.61   22.0     4.8
2         109      ROG Zephyrus G14   Laptop      ASUS  1666.37    8.0     4.8
3          79       RX 9070 XT 16GB      GPU       AMD   617.36   10.0     4.6
4          98     DeathAdder V3 Pro    Mouse     Razer   144.41   12.0     4.6


In [17]:
FEATURE_COLUMNS = [
    "brand",
    "price",
    "stock",
    "rating"
]
TARGET_COLUMN = "category"

print("Feature Columns:", FEATURE_COLUMNS)
print("Target Column:", TARGET_COLUMN)

Feature Columns: ['brand', 'price', 'stock', 'rating']
Target Column: category


In [21]:
x_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df[TARGET_COLUMN].copy()

x_val = val_df[FEATURE_COLUMNS].copy()
y_val = val_df[TARGET_COLUMN].copy()

x_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df[TARGET_COLUMN].copy()

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_val:", x_val.shape)
print("y_val:", y_val.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)

x_train: (150, 4)
y_train: (150,)
x_val: (19, 4)
y_val: (19,)
x_test: (19, 4)
y_test: (19,)


In [22]:
print(x_train.head())
print(y_train.head())

      brand    price  stock  rating
0  Logitech   153.56   21.0     4.8
1        WD    95.61   22.0     4.8
2      ASUS  1666.37    8.0     4.8
3       AMD   617.36   10.0     4.6
4     Razer   144.41   12.0     4.6
0     Mouse
1       SSD
2    Laptop
3       GPU
4     Mouse
Name: category, dtype: object


In [24]:
x_train_encoded = pd.get_dummies(x_train, columns=["brand"], dtype = float)
x_val_encoded = pd.get_dummies(x_val, columns=["brand"], dtype = float)
x_test_encoded = pd.get_dummies(x_test, columns=["brand"], dtype = float)

In [25]:
x_val_encoded = x_val_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)
x_test_encoded = x_test_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)

In [26]:
print("Encoced Features:")
print(x_train_encoded.columns.tolist())
print("\nTrain shape:",x_train_encoded.shape)
print("Validation shape:", x_val_encoded.shape)
print("Test shape:", x_test_encoded.shape)

Encoced Features:
['price', 'stock', 'rating', 'brand_ NVIDIA', 'brand_ Razer', 'brand_ Samsung', 'brand_AMD', 'brand_ASUS', 'brand_Apple', 'brand_Corsair', 'brand_Fractal Design', 'brand_G.Skill', 'brand_Gigabyte', 'brand_HyperX', 'brand_Intel', 'brand_Kingston', 'brand_LG', 'brand_Lian Li', 'brand_Logitech', 'brand_MSI', 'brand_NVIDIA', 'brand_NZXT', 'brand_Razer', 'brand_Samsung', 'brand_Seasonic', 'brand_WD']

Train shape: (150, 26)
Validation shape: (19, 26)
Test shape: (19, 26)


In [27]:
classes = sorted(y_train.unique())
class_to_index = {class_name: index for index, class_name in enumerate(classes)}
index_to_class = {index: class_name for class_name, index in class_to_index.items()}

print("Classes:", classes)
print("\nClass to Index:", class_to_index)

Classes: ['CPU', 'Case', 'GPU', 'Headset', 'Laptop', 'Monitor', 'Motherboard', 'Mouse', 'PSU', 'RAM', 'SSD']

Class to Index: {'CPU': 0, 'Case': 1, 'GPU': 2, 'Headset': 3, 'Laptop': 4, 'Monitor': 5, 'Motherboard': 6, 'Mouse': 7, 'PSU': 8, 'RAM': 9, 'SSD': 10}


In [29]:
y_train_encoded = y_train.map(class_to_index)
y_val_encoded = y_val.map(class_to_index)
y_test_encoded = y_test.map(class_to_index)
print("Y Train:", y_train_encoded)
print("Y Train Dtype:", y_train_encoded.dtype)

Y Train: 0       7
1      10
2       4
3       2
4       7
       ..
145     9
146     2
147    10
148    10
149     1
Name: category, Length: 150, dtype: int64
Y Train Dtype: int64


In [30]:
print("Unknown Validation labels:", y_val_encoded.isna().sum())
print("Unknown Test labels:", y_test_encoded.isna().sum())

Unknown Validation labels: 0
Unknown Test labels: 0


In [38]:
x_train_tensor = torch.tensor(x_train, dtype = torch.float32)
x_val_tensor = torch.tensor(x_val, dtype = torch.float32)
x_test_tensor = torch.tensor(x_test, dtype = torch.float32)

y_train_tensor = torch.tensor(y_train_encoded.to_numpy(), dtype = torch.long)
y_val_tensor = torch.tensor(y_train_encoded.to_numpy(), dtype = torch.long)
y_test_tensor = torch.tensor(y_test_encoded.to_numpy(), dtype = torch.long)

ValueError: could not determine the shape of object type 'DataFrame'

In [35]:
print("x_train:", x_train_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("x_val:", x_val_tensor.shape)
print("y_val:", y_val_tensor.shape)
print("x_test:", x_test_tensor.shape)
print("y_test:", y_test_tensor.shape)

x_train: torch.Size([150])
y_train: torch.Size([150])
x_val: torch.Size([19, 26])


NameError: name 'y_val_tensor' is not defined